In [21]:
import pandas as pd

calls_df = pd.read_csv("../data/ford-catalogue/online_catalogue_annotated.csv")
labels_folder = "../data/ford-catalogue/audacity_emmanuel/processed"

## Adding the raw .txt labels into the DF

In [22]:
import os
from pathlib import Path

# Get all .txt files in labels_folder
txt_files = list(Path(labels_folder).glob("*.txt"))
print(f"Found {len(txt_files)} .txt files")

# Initialize the raw_labels_txt column with None
calls_df['raw_labels_txt'] = None

# Process each .txt file
for txt_file in txt_files:
    # Extract filename without extension
    filename = txt_file.stem
    print(f"Processing: {filename}")
    
    # Check if filename exists in calls_df
    if filename not in calls_df['filename'].values:
        raise ValueError(f"ERROR: No match found for '{filename}' in calls_df")
    
    # Read the file contents
    with open(txt_file, 'r') as f:
        content = f.read()
    
    # Add content to the corresponding row(s) in calls_df
    calls_df.loc[calls_df['filename'] == filename, 'raw_labels_txt'] = content
    print(f"  ✓ Matched and added content ({len(content)} chars)")

print(f"\nSuccessfully processed all files!")
print(f"Rows with raw_labels_txt: {calls_df['raw_labels_txt'].notna().sum()}")

Found 24 .txt files
Processing: N01i-A1-3
  ✓ Matched and added content (211 chars)
Processing: N05ii-B1-1
  ✓ Matched and added content (170 chars)
Processing: N07i-A1-1
  ✓ Matched and added content (75 chars)
Processing: N01iii-C1-1
  ✓ Matched and added content (220 chars)
Processing: N16i-B1-1
  ✓ Matched and added content (236 chars)
Processing: N09ii-A4-1
  ✓ Matched and added content (183 chars)
Processing: N07iii-I1,I2,I18-1
  ✓ Matched and added content (142 chars)
Processing: N08ii-C1-1
  ✓ Matched and added content (170 chars)
Processing: N01ii-B1-1
  ✓ Matched and added content (196 chars)
Processing: N05i-A1-3
  ✓ Matched and added content (55 chars)
Processing: N03-A5-1
  ✓ Matched and added content (175 chars)
Processing: N05i-A1-1
  ✓ Matched and added content (52 chars)
Processing: N08i-A1-2
  ✓ Matched and added content (124 chars)
Processing: N08i-A4-2
  ✓ Matched and added content (64 chars)
Processing: N27-A1-1
  ✓ Matched and added content (153 chars)
Processing:

In [23]:
# Display rows that have raw_labels_txt populated
calls_df[calls_df['raw_labels_txt'].notna()][['filename', 'call_type', 'raw_labels_txt']].head(5)

,filename,call_type,raw_labels_txt
2,N01i-A1-3,N01i,0.000000\t0.066844\tNote: P2 is very unclear\n...
4,N01ii-B1-1,N01ii,0.236344\t0.467889\tP1: VERTICAL\n0.467889\t1....
8,N01iii-C1-1,N01iii,0.349307\t0.647757\tP1: VERTICAL\n0.626343\t1....
14,N02-A4-1,N02,0.432625\t0.543058\tP1: BLUR\n0.530535\t1.1088...
19,N03-A5-1,N03,0.000000\t0.045245\tNote: P3 to be discussed\n...


In [24]:
import re
#TODO: Maybe verify that each PEAK has a max and each VALLEY has a min
def parse_labels(raw_text):
    """
    Parse tab-separated labels from Audacity.
    Returns a dict with parsed labels for P1-P5, BIPHO, EXTRA, note, and point markers (max/min).
    """
    if pd.isna(raw_text):
        return {}
    
    result = {}
    lines = raw_text.strip().split('\n')
    
    # Valid prefixes (case insensitive)
    valid_prefixes = ['P1:', 'P2:', 'P3:', 'P4:', 'P5:', 'BIPHO:', 'EXTRA:', 'NOTE:', 'MAX', 'MIN']
    
    for line_num, line in enumerate(lines, 1):
        line = line.strip()
        if not line:  # Skip empty lines
            continue
        
        # Split by tab
        parts = line.split('\t')
        if len(parts) < 3:
            # Check if this is just a label without timestamps (might be invalid)
            print(f"  ⚠️  Line {line_num}: Not enough parts (expected 3 tab-separated values): {line}")
            continue
        
        start_time = parts[0]
        end_time = parts[1]
        label = parts[2]
        
        # Check if label starts with a valid prefix (case insensitive)
        label_upper = label.upper()
        valid = False
        
        # Check for point markers (max/min, max bipho/min bipho) - these are exact matches
        if label_upper in ['MAX', 'MIN', 'MAX BIPHO', 'MIN BIPHO']:
            valid = True
            # Verify that start and end times are equal (point marker)
            if start_time != end_time:
                raise ValueError(f"Line {line_num}: '{label}' is a point marker but timestamps are not equal: {start_time} != {end_time}")
            
            # Store the time value
            time_value = float(start_time)
            is_bipho = 'BIPHO' in label_upper
            if 'MAX' in label_upper:
                key = 'BIPHO_PEAK_max' if is_bipho else 'PEAK_max'
                result[key] = time_value
            else:  # MIN
                key = 'BIPHO_VALLEY_min' if is_bipho else 'VALLEY_min'
                result[key] = time_value
        else:
            # Check for other prefixes with colons
            for prefix in valid_prefixes:
                if label_upper.startswith(prefix):
                    valid = True
                    # Extract the prefix and content
                    prefix_match = re.match(r'^(P[1-5]|BIPHO|EXTRA|NOTE)\s*:', label, re.IGNORECASE)
                    if prefix_match:
                        prefix_key = prefix_match.group(1).upper()
                        content = label[prefix_match.end():].strip()
                        
                        if prefix_key == 'NOTE':
                            # For notes, only keep content (timestamps don't matter)
                            if 'note' not in result:
                                result['note'] = content
                            else:
                                result['note'] += '; ' + content  # Concatenate multiple notes
                        else:
                            # For P1-P5, BIPHO, EXTRA: keep timestamps and content
                            result[f'{prefix_key}_start'] = float(start_time)
                            result[f'{prefix_key}_end'] = float(end_time)
                            result[prefix_key] = content
                    break
        
        if not valid:
            raise ValueError(f"Line {line_num}: Invalid prefix. Got: {label}, Expected one of {valid_prefixes} ")
    
    return result


In [25]:
# Apply parsing to all rows and create columns
print("\nApplying parsing to dataframe...")

# Initialize all possible columns
label_columns = []
for i in range(1, 6):  # P1-P5
    label_columns.extend([f'P{i}_start', f'P{i}_end', f'P{i}'])
label_columns.extend(['BIPHO_start', 'BIPHO_end', 'BIPHO'])
label_columns.extend(['EXTRA_start', 'EXTRA_end', 'EXTRA'])
label_columns.extend(['note', 'PEAK_max', 'VALLEY_min', 'BIPHO_PEAK_max', 'BIPHO_VALLEY_min'])

# Initialize columns with None
for col in label_columns:
    calls_df[col] = None
calls_df["Annotated"] = False

# Parse and populate
for idx in calls_df.index:
    raw_text = calls_df.loc[idx, 'raw_labels_txt']
    filename = calls_df.loc[idx, 'filename']
    if pd.notna(raw_text):
        # print(f"\nProcessing {filename}:")
        try:
            parsed = parse_labels(raw_text)
            for key, value in parsed.items():
                calls_df.at[idx, key] = value
            calls_df.at[idx, "Annotated"] = True
            # print(f"  ✓ Parsed successfully: {list(parsed.keys())}")
        except Exception as e:
            print(f"  ✗ Error Processing {filename}: {e}")

print(f"✓ Parsing complete!")
print(f"\nColumns added: {label_columns}")
print(f"Rows with parsed labels: {calls_df['P1'].notna().sum()}")


Applying parsing to dataframe...
✓ Parsing complete!

Columns added: ['P1_start', 'P1_end', 'P1', 'P2_start', 'P2_end', 'P2', 'P3_start', 'P3_end', 'P3', 'P4_start', 'P4_end', 'P4', 'P5_start', 'P5_end', 'P5', 'BIPHO_start', 'BIPHO_end', 'BIPHO', 'EXTRA_start', 'EXTRA_end', 'EXTRA', 'note', 'PEAK_max', 'VALLEY_min', 'BIPHO_PEAK_max', 'BIPHO_VALLEY_min']
Rows with parsed labels: 23


In [26]:
# Display the parsed results
parsed_rows = calls_df[calls_df['Annotated']].copy()

# Select relevant columns to display
display_cols = ['filename', 'P1_start', 'P1_end', 'P1', 'P2_start', 'P2_end', 'P2', 
                'P3_start', 'P3_end', 'P3', 'BIPHO', 'PEAK_max', 'VALLEY_min', 'note']
available_cols = [col for col in display_cols if col in parsed_rows.columns]

parsed_rows[available_cols].head(5)



,filename,P1_start,P1_end,P1,P2_start,P2_end,P2,P3_start,P3_end,P3,BIPHO,PEAK_max,VALLEY_min,note
2,N01i-A1-3,0.378784,0.574542,BLUR,0.574542,0.69709,TIGHT FLAT ?,0.682766,1.941666,"PEAK, FLAT","UP, FLAT, DOWN FAST SHORT",0.757568,None,P2 is very unclear
4,N01ii-B1-1,0.236344,0.467889,VERTICAL,0.467889,0.50388,GAP,0.50388,1.325685,"PEAK, FLAT","UP FAST SHORT, PEAK, DOWN SLOW",0.572264,None,None
8,N01iii-C1-1,0.349307,0.647757,VERTICAL,0.647757,0.714674,TIGHT FLAT,0.714674,1.351723,"LEFT PEAK, FLAT","UP FAST SHORT, PEAK, FLAT, DOWN FAST SHORT",0.756162,None,None
14,N02-A4-1,0.432625,0.543058,BLUR,0.530535,1.108886,"SQUIGGLE, FLAT",1.108886,1.169225,UP FAST SHORT,None,None,None,None
19,N03-A5-1,0.162881,0.195306,BLUR,0.200585,0.241305,TIGHT FLAT SHORT,0.245076,0.487135,"LEFT PEAK, DOWN TIGHT",None,0.307664,None,P3 to be discussed


## Validating Vocab


In [27]:
from token_library import VOCAB, ALL_VALID_TOKENS, ALLOWED_PUNCTUATION

In [28]:
def validate_label_content(content, label_prefix, filename):
    """
    Validate that every token in a label's content belongs to the vocabulary.
    
    Content structure: comma-separated units, each unit is space-separated tokens.
    '|' separates alternatives, '?' marks uncertainty — both are allowed punctuation.
    
    Returns a list of error strings (empty if valid).
    """
    errors = []
    content = content.strip()
    if not content:
        return errors
    
    # Split into units (comma-separated sub-parts)
    units = [u.strip() for u in content.split(',')]
    
    for unit_idx, unit in enumerate(units):
        if not unit:
            continue
        # Tokenize: split by whitespace
        raw_tokens = unit.split()
        
        for token in raw_tokens:
            token_upper = token.upper().strip()
            # Skip allowed punctuation
            if token_upper in ALLOWED_PUNCTUATION:
                continue
            # Strip trailing punctuation (e.g. "FLAT?")
            cleaned = token_upper.rstrip('?')
            if not cleaned:
                continue
            if cleaned not in ALL_VALID_TOKENS:
                errors.append(
                    f"[{filename}] {label_prefix} unit {unit_idx+1} \"{unit}\" → "
                    f"unknown token \"{token}\""
                )
    
    return errors


def validate_all_labels(df):
    """
    Validate vocabulary for all Px, BIPHO, and EXTRA columns in the dataframe.
    Returns all errors found.
    """
    all_errors = []
    
    # Columns to validate (content columns, not _start/_end)
    content_cols = [f'P{i}' for i in range(1, 6)] + ['BIPHO', 'EXTRA']
    
    for idx in df[df['Annotated'] == True].index:
        filename = df.loc[idx, 'filename']
        
        for col in content_cols:
            val = df.loc[idx, col]
            if pd.notna(val) and val:
                errors = validate_label_content(str(val), col, filename)
                all_errors.extend(errors)
    
    return all_errors


# Run validation
errors = validate_all_labels(calls_df)

if errors:
    print(f"❌ Found {len(errors)} vocabulary issue(s):\n")
    for err in errors:
        print(f"  • {err}")
else:
    print("✅ All labels pass vocabulary validation!")

✅ All labels pass vocabulary validation!


In [29]:
calls_df.to_csv("../data/ford-catalogue/online_catalogue_annotated_parsed.csv", index=False)